# Scenario
- We are receiving time series data from an application
- We receive thousands of data points per day
- Initially it makes sense to partition the data by each day
- But we will see how partition is not helpful always
- How Liquid Clustering can be better strategy than Partition

In [0]:
from pyspark.sql.functions import *
from datetime import datetime, timedelta
from pyspark.sql.types import *

# Generate Data

In [0]:
start_date = datetime(2025, 1, 1)
end_date = datetime(2026, 1, 1)
date_list = [start_date + timedelta(days=x) for x in range((end_date - start_date).days + 1)]
unix_times = [int(dt.timestamp()) for dt in date_list]

schema = StructType([
    StructField('time', TimestampType(), True),
    StructField('id', IntegerType(), True),
    StructField('value', FloatType(), True),
])

df_full = spark.createDataFrame(data=[],schema=schema)
data_point_per_day = 10000
                           
for i in unix_times:
    df = (
    spark.range(0,data_point_per_day)
    .select(
        from_unixtime(lit(i + col("id"))).cast('timestamp').alias("time"),
        hash("id").alias("id"),
        rand().alias("value"),      
        )
    )
    df_full = df_full.unionByName(df)

df_full.display()

time,id,value
2025-01-01T00:00:00.000Z,-1670924195,0.11910636493801507
2025-01-01T00:00:01.000Z,-1712319331,0.827353930386264
2025-01-01T00:00:02.000Z,-797927272,0.6738796872158781
2025-01-01T00:00:03.000Z,519220707,0.9379036010724543
2025-01-01T00:00:04.000Z,1344313940,0.5654863195059767
2025-01-01T00:00:05.000Z,1607884268,0.6061609899845574
2025-01-01T00:00:06.000Z,-1767354555,0.8274430311456822
2025-01-01T00:00:07.000Z,1293116811,0.3060820455995432
2025-01-01T00:00:08.000Z,-1131184084,0.3421183775037876
2025-01-01T00:00:09.000Z,1504843649,0.989085621078749


# Phase 1: Save Partition By Date

In [0]:
(
    df_full
    .withColumn('event_date',date_trunc('day',col('time')))
    .write
    .mode('overwrite')
    .option('overwriteSchema','true')
    .partitionBy('event_date')
    .saveAsTable('telemetry.iot_telemetry')
)

# Run Optimize

In [0]:
%sql
OPTIMIZE telemetry.iot_telemetry

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 366, null, null, 0, 0, 366, 366, true, 0, 0, 1772179901977, 1772179904520, 8, 0, null, List(0, 0), null, 4, 4, 0, 0, null, null)"


# Run a random Query

In [0]:
%sql
SELECT AVG(value) 
FROM telemetry.iot_telemetry
WHERE `time` >= "2025-02-01T07:02:43.000+00:00" 
AND `time` <= "2025-10-11T07:29:15.000+00:00"

AVG(value)
0.4999797102051497


# Analysis Query Profiling
<br>
<img src="./image_1772178880144.png" alt="image_1772178880144.png" width="400" height="250"/>
<br>

- The above query took 7 seconds to finish
- 252 files were opened, loaded in memory, and closed to execute this query
- That is lot of overhead

# Pase 2: Save without partition

In [0]:
(
    df_full
    .withColumn('event_date',date_trunc('day',col('time')))
    .write
    .mode('overwrite')
    .option('overwriteSchema','true')
    .saveAsTable('telemetry.iot_telemetry_unpartitioned')
)

# Liquid Cluster and Optimize

In [0]:
%sql
ALTER TABLE telemetry.iot_telemetry_unpartitioned CLUSTER BY AUTO;
OPTIMIZE telemetry.iot_telemetry_unpartitioned;

path,metrics
,"List(1, 2547, List(29650483, 29650483, 2.9650483E7, 1, 29650483), List(20049, 1375724, 21669.514330585003, 2547, 55192253), 0, null, null, 0, 1, 2547, 0, true, 0, 0, 1772180213019, 1772180228142, 8, 1, null, List(0, 0), null, 4, 4, 7224, 0, null, null)"


# Run same query

In [0]:
%sql
SELECT AVG(value) 
FROM telemetry.iot_telemetry_unpartitioned
WHERE `time` >= "2025-02-01T07:02:43.000+00:00" 
AND `time` <= "2025-10-11T07:29:15.000+00:00"

AVG(value)
0.5001345618776033


# Analysis Query Profiling
<img src="./image_1772180299360.png" alt="image_1772180299360.png" width="400" height="250"/>
<br><br>

- The above query took 2 seconds to finish
- Only one file is read
- Databricks auto optimized the table and underlying files

# Conclusion
- Liquid Clustering is highly optimized
- Optimize command compacts and removes unnecessary files
- Partition By should be used carefully
- We saw user might not always use the partition columns to search data